In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/dataset_final.csv')
df.head()
display(df)

,text,label
0,Cán bộ công an tử vong vì tai nạn giao thông h...,1
1,Chủ tịch UBND TP HCM: Chọn 38 trung tâm giải q...,1
2,Dùng cần cẩu giải cứu 2 người đi xe máy rơi xu...,1
3,Khói bụi trắng bao trùm khu vực gần nhà máy th...,1
4,Công bố quy trình 5 bước bổ nhiệm công chức là...,1
...,...,...
46831,BRUSSELS (Reuters) - Các đồng minh NATO hôm th...,1
46832,"LONDON (Reuters) - LexisNexis, nhà cung cấp th...",1
46833,MINSK (Reuters) - Dưới bóng các nhà máy thời X...,1
46834,MOSCOW (Reuters) - Bộ trưởng Ngoại giao Vatica...,1


In [ ]:
print(df['text'][16])

Lòng tốt luôn ở quanh ta. Trong cuộc sống bộn bề lo toan, những câu chuyện về lòng tử tế luôn là ngọn lửa sưởi ấm tâm hồn và lan tỏa năng lượng tích cực Sự tử tế trong guồng quay hối hả của cuộc sống không chỉ là những hành động lớn lao mà đôi khi là sự quan tâm, sẻ chia với những cảnh đời kém may mắn. Gánh bánh mì yêu thương Cụ Nguyễn Thị Ngang (phường Phú An, TP HCM) được nhiều người trìu mến gọi với cái tên thân thương ngoại Sáu". Gần 40 năm qua


In [ ]:
from pathlib import Path
import math
import re
from collections import defaultdict
import numpy as np

In [ ]:
pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.8/657.8 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.7 MB/s eta 0:00:00


In [ ]:
import regex as re

from underthesea import word_tokenize
EMAIL = re.compile(r"([\w0-9_\.-]+)(@)([\d\w\.-]+)(\.)([\w\.]{2,6})")
URL = re.compile(r"https?:\/\/(?!.*:\/\/)\S+")
PHONE = re.compile(r"(09|01[2|6|8|9])+([0-9]{8})\b")
MENTION = re.compile(r"@.+?:")
NUMBER = re.compile(r"\d+.?\d*")
DATETIME = '\d{1,2}\s?[/-]\s?\d{1,2}\s?[/-]\s?\d{4}'

RE_HTML_TAG = re.compile(r'<[^>]+>')
RE_CLEAR_1 = re.compile("[^_<>\s\p{Latin}]")
RE_CLEAR_2 = re.compile("__+")
RE_CLEAR_3 = re.compile("\s+")



def replace_common_token(txt):
    txt = re.sub(EMAIL, ' ', txt)
    txt = re.sub(URL, ' ', txt)
    txt = re.sub(MENTION, ' ', txt)
    txt = re.sub(DATETIME, ' ', txt)
    txt = re.sub(NUMBER, ' ', txt)
    return txt


def remove_emoji(txt):
    txt = re.sub(':v', '', txt)
    txt = re.sub(':D', '', txt)
    txt = re.sub(':3', '', txt)
    txt = re.sub(':\(', '', txt)
    txt = re.sub(':\)', '', txt)
    return txt


def remove_html_tag(txt):
    return re.sub(RE_HTML_TAG, ' ', txt)

def preprocess(txt, tokenize=True):
    txt = remove_html_tag(txt)
    txt = re.sub('&.{3,4};', ' ', txt)

    txt = txt.lower()
    txt = replace_common_token(txt)
    txt = remove_emoji(txt)
    txt = re.sub(RE_CLEAR_1, ' ', txt)
    txt = re.sub(RE_CLEAR_2, ' ', txt)
    txt = re.sub(RE_CLEAR_3, ' ', txt)
    txt = txt.strip()
    if tokenize:
      txt = word_tokenize(txt)
    return " ".join(txt)

In [ ]:

listDocs = []
for doc in df['text']:
    listDocs.append(preprocess(str(doc)))

In [ ]:
print(df['label'].value_counts())

label
1    25408
0    21428
Name: count, dtype: int64


In [ ]:
from transformers import AutoTokenizer
import torch
from tqdm import tqdm


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

# Giả sử df đã có:
# df = DataFrame có N dòng, cột cuối là nhãn

# Lấy cột nhãn từ df (giả sử là cột cuối)
labels = df.iloc[:, -1].values  # hoặc df[df.columns[-1]]

# Tạo DataFrame mới
df = pd.DataFrame({
    'text': listDocs,
    'label': labels
})

# df = pd.DataFrame(data)


In [ ]:
#class PhoBERT
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW  # Sử dụng bản gốc từ PyTorch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd


# Dùng GPU nếu có
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load tokenizer và model PhoBERT
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

model2 = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base", num_labels=2).to(device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

In [ ]:
class PhoBERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoded.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

final_dataset = PhoBERTDataset(df['text'].tolist(), df['label'].tolist(), tokenizer)
final_loader = DataLoader(final_dataset, batch_size=16, shuffle=True)

In [ ]:
optimizer2 = AdamW(model2.parameters(), lr=2e-5)

model2.train()
for epoch in range(5):
    total_loss = 0
    for batch in final_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model2(**batch)
        loss = outputs.loss

        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss / len(final_loader):.4f}")


Epoch 1: Loss = 0.0958
Epoch 2: Loss = 0.0487
Epoch 3: Loss = 0.0335
Epoch 4: Loss = 0.0232
Epoch 5: Loss = 0.0163


In [ ]:
# model2.eval()
# all_preds, all_labels = [], []

# with torch.no_grad():
#     for batch in test_loader:
#         labels = batch['labels'].numpy()
#         batch = {k: v.to(device) for k, v in batch.items()}
#         outputs = model2(**batch)
#         logits = outputs.logits
#         preds = torch.argmax(logits, dim=1).cpu().numpy()

#         all_preds.extend(preds)
#         all_labels.extend(labels)

# print(classification_report(all_labels, all_preds))
save_path = "/content/drive/MyDrive/phobert_fakenews_model"
model2.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/phobert_fakenews_model/tokenizer_config.json',
 '/content/drive/MyDrive/phobert_fakenews_model/special_tokens_map.json',
 '/content/drive/MyDrive/phobert_fakenews_model/vocab.txt',
 '/content/drive/MyDrive/phobert_fakenews_model/bpe.codes',
 '/content/drive/MyDrive/phobert_fakenews_model/added_tokens.json')